# k-Nearest Neighbours — learning by memorizing

> Tutorial pair for [`knn.py`](knn.py).

## 1. Intuition
"You are the average of your k closest friends." To label a new point, find the
k training points nearest to it and let them vote. There is **no training** — we
just store the data and do all the work at query time (*lazy learning*).

## 2. Concept (the slide)
- **Classification:** majority (optionally distance-weighted) vote of the k nearest.
- **Regression:** (weighted) mean of the k nearest targets.
- **Hyperparameters:** $k$, the distance metric, and the weighting.
- **Geometry matters:** features must be on comparable scales (standardize!), and
  it degrades in high dimensions (curse of dimensionality).

## 3. Math derivation

**Distance.** Minkowski-$p$: $\;d_p(\mathbf a,\mathbf b)=\big(\sum_j|a_j-b_j|^p\big)^{1/p}$
($p{=}2$ Euclidean, $p{=}1$ Manhattan). For Euclidean we use the identity
$\lVert\mathbf a-\mathbf b\rVert^2=\lVert\mathbf a\rVert^2+\lVert\mathbf b\rVert^2-2\,\mathbf a^\top\mathbf b$
to compute all pairwise distances with one matmul.

**Prediction.** Let $\mathcal N_k(\mathbf x)$ be the indices of the k nearest
neighbours and $w_i$ a weight ($w_i{=}1$ uniform, or $w_i{=}1/d_i$ weighted).
$$\hat y_{\text{clf}}=\arg\max_{c}\sum_{i\in\mathcal N_k}w_i\,\mathbb 1[y_i=c],
  \qquad
  \hat y_{\text{reg}}=\frac{\sum_{i\in\mathcal N_k}w_i\,y_i}{\sum_{i\in\mathcal N_k}w_i}.$$

**Bias–variance via $k$.** Small $k$ → flexible, low bias, **high variance**
(noisy boundary). Large $k$ → smooth, higher bias, low variance. The
1-NN error is famously $\le 2\times$ the Bayes error as $n\to\infty$
(Cover & Hart).

**Speed.** Brute force is $O(nd)$ per query. A **KD-tree** partitions space so
the average query is $O(\log n)$ in low dimensions (it prunes whole subtrees
whose bounding region is farther than the current k-th best).

## 4. NumPy implementation (brute force + KD-tree)

In [ ]:
# ===== actual implementation from knn.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _pairwise(A, B, p=2):
    """||a - b||_p for every pair. A:(n,d) B:(m,d) -> (n,m)."""
    if p == 2:  # ||a-b||^2 = |a|^2 + |b|^2 - 2 a·b  (fast, vectorized)
        a2 = (A**2).sum(1)[:, None]
        b2 = (B**2).sum(1)[None, :]
        d2 = np.maximum(a2 + b2 - 2 * A @ B.T, 0.0)
        return np.sqrt(d2)
    return np.power(np.abs(A[:, None, :] - B[None, :, :]) ** p, 1).sum(2) ** (1 / p)

import torch

def demo():
    np.random.seed(SEED)
    from sklearn.datasets import make_moons

    X, y = make_moons(n_samples=300, noise=0.25, random_state=SEED)
    mu, sd = X.mean(0), X.std(0); X = (X - mu) / sd
    n_tr = 220
    Xtr, ytr, Xte, yte = X[:n_tr], y[:n_tr], X[n_tr:], y[n_tr:]

    for k in (1, 5, 15):
        m = KNNNumPy(k=k).fit(Xtr, ytr)
        acc = np.mean(m.predict(Xte) == yte)
        print(f"k={k:2d}  test acc={acc:.3f}")

    acc_w = np.mean(KNNNumPy(k=15, weights="distance").fit(Xtr, ytr).predict(Xte) == yte)
    print(f"k=15 distance-weighted acc={acc_w:.3f}")

    acc_t = np.mean(knn_torch(Xtr, ytr, Xte, k=5) == yte)
    print(f"torch kNN (k=5) acc={acc_t:.3f}")

    # KD-tree vs brute force agree on nearest neighbour
    tree = KDTree(Xtr)
    q = Xte[0]
    kd = [i for _, i in tree.query(q, k=3)]
    bf = np.argsort(_pairwise(q[None], Xtr)[0])[:3].tolist()
    print(f"KD-tree NN {kd}  ==  brute force {bf}  -> {set(kd) == set(bf)}")


class KNNNumPy:
    def __init__(self, k=5, task="classification", weights="uniform", p=2):
        self.k, self.task, self.weights, self.p = k, task, weights, p

    def fit(self, X, y):                 # "training" = remember the data
        self.X = np.asarray(X, float)
        self.y = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        D = _pairwise(X, self.X, self.p)             # (n_query, n_train)
        idx = np.argpartition(D, self.k, axis=1)[:, :self.k]   # k smallest
        neigh_y = self.y[idx]
        neigh_d = np.take_along_axis(D, idx, axis=1)
        if self.weights == "distance":
            w = 1.0 / (neigh_d + 1e-12)
        else:
            w = np.ones_like(neigh_d)

        out = []
        for i in range(len(X)):
            if self.task == "classification":
                classes = np.unique(neigh_y[i])
                scores = {c: w[i][neigh_y[i] == c].sum() for c in classes}
                out.append(max(scores, key=scores.get))
            else:  # regression: weighted average
                out.append(np.average(neigh_y[i], weights=w[i]))
        return np.array(out)


class KDTree:
    """Minimal KD-tree for exact nearest-neighbour search in low dimensions."""

    class _Node:
        __slots__ = ("point", "idx", "axis", "left", "right")

    def __init__(self, X):
        self.X = np.asarray(X, float)
        self.root = self._build(np.arange(len(X)), depth=0)

    def _build(self, idx, depth):
        if len(idx) == 0:
            return None
        axis = depth % self.X.shape[1]
        order = idx[np.argsort(self.X[idx, axis])]
        mid = len(order) // 2
        node = self._Node()
        node.idx = order[mid]; node.point = self.X[order[mid]]; node.axis = axis
        node.left = self._build(order[:mid], depth + 1)
        node.right = self._build(order[mid + 1:], depth + 1)
        return node

    def query(self, q, k=1):
        import heapq
        heap = []  # max-heap of (-dist, idx)

        def visit(node):
            if node is None:
                return
            d = np.linalg.norm(q - node.point)
            if len(heap) < k:
                heapq.heappush(heap, (-d, int(node.idx)))
            elif d < -heap[0][0]:
                heapq.heapreplace(heap, (-d, int(node.idx)))
            diff = q[node.axis] - node.point[node.axis]
            near, far = (node.left, node.right) if diff < 0 else (node.right, node.left)
            visit(near)
            if len(heap) < k or abs(diff) < -heap[0][0]:   # hypersphere crosses plane
                visit(far)

        visit(self.root)
        return sorted([(-nd, i) for nd, i in heap])

## 5. PyTorch implementation (`torch.cdist`, GPU-ready)

In [ ]:
# ===== actual implementation from knn.py =====
def knn_torch(X_train, y_train, X_query, k=5, task="classification"):
    """Vectorized kNN with torch.cdist (works on GPU)."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    Xt = torch.as_tensor(X_train, dtype=torch.float32, device=dev)
    yt = torch.as_tensor(y_train, device=dev)
    Xq = torch.as_tensor(X_query, dtype=torch.float32, device=dev)
    D = torch.cdist(Xq, Xt)                       # (q, n)
    idx = D.topk(k, largest=False).indices        # k nearest
    neigh = yt[idx]
    if task == "classification":
        out = torch.mode(neigh, dim=1).values
    else:
        out = neigh.float().mean(1)
    return out.cpu().numpy()

## 6. Train — vary k, weighting, and check KD-tree == brute force

In [ ]:
demo()

## 7. Visualization — how k smooths the decision boundary

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import knn as M

X, y = make_moons(n_samples=300, noise=0.25, random_state=0)
X = (X - X.mean(0)) / X.std(0)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200),
                     np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, k in zip(axes, (1, 5, 25)):
    zz = M.KNNNumPy(k=k).fit(X, y).predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.4, cmap="coolwarm")
    ax.scatter(X[:,0], X[:,1], c=y, s=12, edgecolor="k", cmap="coolwarm")
    ax.set_title(f"k = {k}")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Standardize features** — kNN is pure geometry.
- Choose $k$ by cross-validation; odd $k$ avoids ties in binary problems.
- Memory- and query-heavy at scale; use KD/ball-trees or approximate NN.
- Curse of dimensionality: in high-$d$ all points become equidistant.